# PINN4SOH 模块 6：评估与可视化

### 数据流中的位置

```
[模块5: 训练循环]
    │
    ├── pinn.Test(testloader) → true_label, pred_label
    │
    ▼
[模块6: 本模块 — 评估与可视化]
    │
    ├── eval_metrix() → MAE / MAPE / MSE / RMSE
    ├── eval_per_battery() → 每节电池独立评估
    ├── Figure 4a: 预测 vs 真实 SOH 散点图
    ├── Figure 4b: SOH 退化曲线 (每电池)
    └── 训练历史可视化 (Loss + LR 曲线)
```

---

## 复现运行与源码对齐说明

> 当前 clean 代码：`src/06_eval.py`  
> 原始代码：`utils/util.py L42-L48`  
> 执行规范：paper-reproduction；原始仓库只读。  
> 本手册已完成 Notebook JSON 与代码单元静态检查。涉及数据的单元需按 `DATA.md` 准备数据后，从头运行以生成本机结果。

本手册保留六层学习结构：纯净代码、逐行详解、语法表、数据流角色、物理含义与真实数据图、踩坑记录。论文与源码不一致处以源码为复现基准并单独说明。

### 背景：评估 PINN 和评估普通回归模型有什么不同？

**普通回归**：看 MAE/RMSE 就够了——"预测有多准"。

**PINN 评估的额外维度**：
1. **时序一致性**：预测的退化曲线是否平滑？有没有不合理的跳变？
2. **物理一致性**：SOH 是否单调递减（除容量回升外）？
3. **跨电池泛化**：在训练电池上的精度 vs 测试电池上的精度差距有多大？
4. **退化趋势**：模型是否准确捕捉到退化加速（拐点）？

这就要求评估不能只看一个数字——需要**图表**来揭示数字背后的退化行为。

### 模块 6 核心概念概览

> 行号记录自手册编写版本，后续整理可能产生偏移；核对实现时请以函数名和当前 `src/` 文件为准。

| 子模块 | clean_code 行号 | 原始代码行号 | 核心功能 |
|--------|:-----------:|:----------:|----------|
| 6a: eval_metrix | L95-109 | util.py L42-48 | MAE/MAPE/MSE/RMSE 四大指标 |
| 6b: 按电池拆分 | L115-158 | XJTU results.py L97-140 | 时序拆分 + 每电池独立评估 |
| 6c: Figure 4a | L164-210 | — | 预测 vs 真实散点图 |
| 6d: Figure 4b | L216-265 | — | SOH 退化曲线 |
| 6e: 训练历史 | L270-330 | — | Loss + LR 曲线 |

### 目录

| 章节 | 内容 |
|------|------|
| [1. eval_metrix — 四大指标](#sec1) | 每个指标的公式 + 物理含义 |
| [2. 按电池拆分评估](#sec2) | 时序拆分逻辑 + 每电池指标 |
| [3. Figure 4a: 预测 vs 真实](#sec3) | 散点图 + 误差带 |
| [4. Figure 4b: 退化曲线](#sec4) | 每电池的 SOH 轨迹 |
| [5. 训练历史可视化](#sec5) | Loss + LR 曲线 |

---

## 1. 6a: eval_metrix — 四大评估指标 <a id="sec1"></a>

> **对应 src/06_eval.py 第 95-109 行**
> **对应原始代码 utils/util.py 第 42-48 行**

一句话：MAE 看平均偏差，MAPE 看相对偏差，MSE 惩罚大误差，RMSE 最直观（与 SOH 同单位）。

### 1.1 代码块

In [ ]:
from sklearn import metrics
import numpy as np

def eval_metrix(true_label, pred_label):
    """
    四个回归评估指标

    MAE  = mean(|pred - true|)           — 平均绝对误差
    MAPE = mean(|pred - true| / true)    — 平均百分比误差
    MSE  = mean((pred - true)^2)         — 均方误差
    RMSE = sqrt(MSE)                     — 均方根误差
    """
    MAE  = metrics.mean_absolute_error(true_label, pred_label)
    MAPE = metrics.mean_absolute_percentage_error(true_label, pred_label)
    MSE  = metrics.mean_squared_error(true_label, pred_label)
    RMSE = np.sqrt(MSE)
    return [MAE, MAPE, MSE, RMSE]

### 1.1.1 指标解读

| 指标 | 公式 | 单位 | 优秀值 | 物理含义 |
|------|------|:---:|:------:|----------|
| MAE | mean(\|pred-true\|) | SOH | < 0.01 | 平均每个预测偏离真实值 1% SOH |
| MAPE | mean(\|pred-true\|/true)×100 | % | < 1.5% | 相对误差 1.5% 意味着预测 0.80 时误差约 0.012 |
| MSE | mean((pred-true)²) | SOH² | < 0.0002 | 平方后放大大误差，对异常预测敏感 |
| RMSE | sqrt(MSE) | SOH | < 0.015 | 与 SOH 同单位，最直观："典型误差约 1.5% SOH" |

**MAE vs RMSE 的选择**：
- MAE 对所有误差一视同仁（线性）
- RMSE 对大的单点误差更敏感（平方放大）
- 如果模型偶尔有"离谱"的预测 → RMSE >> MAE（说明有大误差）
- 如果 MAE ≈ RMSE → 误差分布均匀，没有特别离谱的点

**MAPE 的陷阱**：
- 真实 SOH = 0.01 时，预测 0.02 的 MAPE = 100%（除以小值爆炸）
- 电池数据 SOH ∈ [0.6, 1.0] 避开了这个问题
- 但晚期 SOH=0.7 时的 MAPE 会比早期 SOH=0.95 时高

## 2. 6b: 按电池拆分评估 <a id="sec2"></a>

> **对应 src/06_eval.py 第 115-158 行**

一句话：整体 MAE 可能隐藏"某些电池预测很差"的事实。按电池拆分才能暴露问题。

### 2.1 拆分逻辑

多节电池的测试数据是这样的排列：

```
[电池A: SOH从0.95→0.80, 180个点] [电池B: SOH从1.0→0.75, 250个点] [电池C: ...]
                                   ↑
                              SOH 突然跳回 1.0
                              → 这就是拆分点!
```

**如何找到拆分点？**
```python
diff = np.diff(true_label)           # 相邻两点的 SOH 变化
split_points = np.where(diff > 0.05) # SOH 突增 > 0.05 → 新电池开始
```

电池退化时 SOH 每步只降 ~0.001。如果突然跳升 0.05+，只能是新电池的数据开始了。

In [ ]:
import sys, os, importlib.util
import numpy as np
CLEAN = r'../src'
sys.path.insert(0, CLEAN)

def _load(fname, mname):
    spec = importlib.util.spec_from_file_location(mname, os.path.join(CLEAN, fname))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

_eval = _load('06_eval.py', 'eval_06')
eval_metrix = _eval.eval_metrix
eval_per_battery = _eval.eval_per_battery
print_eval_report = _eval.print_eval_report

# 用未训练模型推理
_model = _load('03_model.py', 'model_03')
_dataloader = _load('02_dataloader.py', 'dataloader_02')
PINN = _model.PINN
device = _model.device
build_dataloaders = _dataloader.build_dataloaders

ROOT = r'../data/XJTU data'
all_files = [f for f in os.listdir(ROOT) if '2C' in f]
test_files = [os.path.join(ROOT, f) for f in all_files if '4' in f or '8' in f]
test_dict = build_dataloaders(test_files, nominal_capacity=2.0, batch_size=128)

pinn = PINN(F_layers_num=3, F_hidden_dim=60)
true_label, pred_label = pinn.Test(test_dict['test'])

# 按电池拆分
per_battery = eval_per_battery(true_label, pred_label)
print(f"总数据点: {len(true_label)}")
print(f"拆分为 {len(per_battery['MAE_list'])} 节电池")
print()

# 每节电池的指标
print(f"{'电池':<8} {'SOH范围':<16} {'样本数':<8} {'MAE':<10} {'RMSE':<10}")
print('-' * 52)
for i in range(len(per_battery['MAE_list'])):
    true_i = per_battery['true_list'][i]
    print(f"Battery {i+1:<2}  [{true_i.min():.3f}, {true_i.max():.3f}]     "
          f"{len(true_i):<8} {per_battery['MAE_list'][i]:.4f}     {per_battery['RMSE_list'][i]:.4f}")
print()
print("注意: 用未训练模型推理，MAE/RMSE 很高是正常的。训练后应在 ~0.01 量级。")

## 3. 6c: Figure 4a — 预测 vs 真实散点图 <a id="sec3"></a>

> **对应 src/06_eval.py 第 164-210 行**

一句话：所有测试样本的 (真实 SOH, 预测 SOH) 画成散点图。理想状态：所有点在 y=x 对角线上。

### 3.1 图表元素说明

| 元素 | 含义 | 怎么看 |
|------|------|--------|
| 蓝色散点 | 每个样本的 (true, pred) | 离 y=x 越近越好 |
| 红色虚线 (y=x) | 完美预测线 | 点在线上 = 完全准确 |
| 灰色带 (±5%) | 5% 误差范围 | 大部分点应在带内 |
| 左上角文字框 | MAE/MAPE/RMSE | 量化预测精度 |

**合格的 Figure 4a 是什么样的？**
- 绝大多数点落在 ±5% 灰色带内
- MAE < 0.02, RMSE < 0.03
- 晚期 (SOH < 0.8) 的点可能略微偏离（晚期退化更剧烈，更难预测）

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

true_label = np.asarray(true_label).reshape(-1)
pred_label = np.asarray(pred_label).reshape(-1)

fig, ax = plt.subplots(figsize=(7, 7))

ax.scatter(true_label, pred_label, alpha=0.4, s=8,
           c='steelblue', edgecolors='none', label=f'n={len(true_label)} points')

lims = [min(true_label.min(), pred_label.min()) - 0.02,
        max(true_label.max(), pred_label.max()) + 0.02]
ax.plot(lims, lims, 'r--', linewidth=1.2, alpha=0.7, label='y = x (ideal)')

ax.fill_between(lims,
                [l * 0.95 for l in lims], [l * 1.05 for l in lims],
                alpha=0.08, color='gray', label=r'$\pm$5% band')

MAE, MAPE, MSE, RMSE = eval_metrix(true_label, pred_label)
textstr = f'MAE = {MAE:.4f}\nMAPE = {MAPE:.2f}%\nRMSE = {RMSE:.4f}'
props = dict(boxstyle='round,pad=0.3', facecolor='wheat', alpha=0.7)
ax.text(0.05, 0.95, textstr, transform=ax.transAxes,
        fontsize=11, verticalalignment='top', bbox=props)

ax.set_xlabel('True SOH', fontsize=13)
ax.set_ylabel('Predicted SOH', fontsize=13)
ax.set_title('PINN4SOH: Predicted vs True SOH', fontsize=14)
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

IMAGE_DIR = r'../outputs/figures'
os.makedirs(IMAGE_DIR, exist_ok=True)
plt.savefig(os.path.join(IMAGE_DIR, 'fig4a_pred_vs_true.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure 4a saved. MAE={MAE:.4f}, RMSE={RMSE:.4f}")

## 4. 6d: Figure 4b — SOH 退化曲线 <a id="sec4"></a>

> **对应 src/06_eval.py 第 216-265 行**

一句话：每节电池画一条 SOH vs Cycle 曲线。蓝色实线=真实退化轨迹，红色虚线=模型预测轨迹。两条线越接近越好。

### 4.1 怎么看退化曲线

| 观察点 | 怎么看 | 好模型 | 差模型 |
|--------|--------|--------|--------|
| **整体偏移** | 预测线是整体高于还是低于真实线？ | 交错缠绕 | 系统性地偏离 (如总是偏高 0.02) |
| **趋势跟踪** | 预测线是否跟随真实线的拐点？ | 拐点一致 | 一条直线穿过曲线变化 |
| **容量回升** | 真实线有回升时，预测线是否也回升？ | 同步小幅度回升 | 忽略回升 或 过度放大回升 |
| **晚期发散** | SOH < 0.8 时两线是否分叉？ | 保持接近 | 明显分叉 (晚期预测不准) |
| **异常跳变** | 预测线有无不合理的突变？ | 平滑 | 锯齿状或台阶状突变 |

In [ ]:
per_battery = eval_per_battery(true_label, pred_label)
n_batteries = min(len(per_battery['MAE_list']), 4)

cols = min(2, n_batteries)
rows = (n_batteries + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 4*rows))
if rows * cols == 1:
    axes = np.array([axes])
axes = axes.flatten()

for i in range(n_batteries):
    ax = axes[i]
    true_i = per_battery['true_list'][i]
    pred_i = per_battery['pred_list'][i]
    cycles = np.arange(len(true_i))

    ax.plot(cycles, true_i, 'b-', linewidth=1.2, alpha=0.8, label='True')
    ax.plot(cycles, pred_i, 'r--', linewidth=1.0, alpha=0.8, label='Predicted')

    ax.set_title(f'Battery {i+1} | MAE={per_battery["MAE_list"][i]:.4f} '
                 f'RMSE={per_battery["RMSE_list"][i]:.4f}', fontsize=10)
    ax.set_xlabel('Cycle', fontsize=9)
    ax.set_ylabel('SOH', fontsize=9)
    ax.legend(fontsize=8, loc='lower left')
    ax.grid(True, alpha=0.3)

for i in range(n_batteries, len(axes)):
    axes[i].set_visible(False)

fig.suptitle('PINN4SOH: SOH Degradation Curves', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(IMAGE_DIR, 'fig4b_degradation_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Figure 4b saved (未训练模型, 仅验证流程).")

## 5. 6e: 训练历史可视化 <a id="sec5"></a>

> **对应 src/06_eval.py 第 270-330 行**

一句话：画出训练过程中 Loss 的下降曲线和学习率的变化曲线。这是诊断训练问题的第一手资料。

### 5.1 怎么看训练历史图

**Loss 曲线 (左上)**：
- 三条线都在下降 → 训练正常
- L_data 下降但 L_PDE 上升 → 模型在"牺牲物理一致性来拟合数据"→ 考虑增大 α
- L_mono 始终为 0 → 模型学会了不违反单调性（好！）或 batch 太小没遇上违规（调大 batch_size）
- 剧烈震荡 → 学习率太大，考虑减小 base_lr 或延长 warmup

**学习率曲线 (右上)**：
- warmup 期线性上升 → 正常
- cosine 衰减平滑 → 正常
- 竖线标记 warmup 结束点

**验证 MSE (左下)**：
- 单调下降 → 训练正常
- 下降到某点后回升 → 过拟合！需要更早的 early stop 或更大的 dropout
- 从未下降 → 模型没学到东西，检查数据和模型架构

**测试 RMSE (右下)**：
- 每次最佳模型更新时记录一次
- 持续下降 → 模型在持续进步
- 稳定不变 → 验证集不再改善，early stop 即将触发

In [ ]:
# 模拟一段训练历史来展示图表结构
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
epochs_arr = np.arange(1, 81)

# 模拟数据
train_l1 = 0.1 * np.exp(-0.04 * epochs_arr) + 0.005 * np.random.randn(80) + 0.005
train_l2 = 0.2 * np.exp(-0.03 * epochs_arr) + 0.01 * np.random.randn(80) + 0.01
train_l3 = 0.01 * np.exp(-0.06 * epochs_arr) + 0.001 * np.random.randn(80)
train_l3 = np.maximum(train_l3, 0)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss
ax = axes[0, 0]
ax.plot(epochs_arr, train_l1, 'b-', linewidth=1, alpha=0.8, label='L_data')
ax.plot(epochs_arr, train_l2, 'r-', linewidth=1, alpha=0.8, label='L_PDE')
ax.plot(epochs_arr, train_l3, 'g-', linewidth=1, alpha=0.8, label='L_mono')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Training Loss Components')
ax.legend(); ax.grid(True, alpha=0.3)

# LR
warmup, base, final = 30, 0.01, 0.0002
lrs = np.concatenate([
    np.linspace(0.002, base, warmup),
    final + 0.5*(base-final)*(1+np.cos(np.pi*np.arange(80-warmup)/(80-warmup)))
])
ax = axes[0, 1]
ax.plot(epochs_arr, lrs, 'purple', linewidth=1.5)
ax.axvline(x=warmup, color='red', linestyle='--', alpha=0.4, label='warmup end')
ax.set_xlabel('Epoch'); ax.set_ylabel('LR')
ax.set_title('Learning Rate Schedule'); ax.legend(); ax.grid(True, alpha=0.3)

# Valid MSE
valid_mse = 0.3 * np.exp(-0.04 * epochs_arr) + 0.005 * np.random.randn(80) + 0.01
valid_mse[60:] += np.linspace(0, 0.01, 20)
ax = axes[1, 0]
ax.plot(epochs_arr, valid_mse, 'b-', linewidth=1)
best_idx = np.argmin(valid_mse)
ax.axvline(x=best_idx+1, color='green', linestyle='--', alpha=0.5, label=f'best epoch {best_idx+1}')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE')
ax.set_title('Validation MSE (with overfitting after epoch 60)')
ax.legend(); ax.grid(True, alpha=0.3)

# Test RMSE
test_rmse = [0.08, 0.05, 0.035, 0.028, 0.025, 0.023, 0.022, 0.021, 0.020]
ax = axes[1, 1]
ax.plot(range(1, len(test_rmse)+1), test_rmse, 'ro-', markersize=5, linewidth=1)
ax.set_xlabel('Best Model Update #')
ax.set_ylabel('RMSE')
ax.set_title('Test RMSE at Each Best-Model Update')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(IMAGE_DIR, 'training_history_demo.png'), dpi=150, bbox_inches='tight')
plt.show()
print("训练历史 demo 图已保存。")

---

### 与原始代码的对应关系

| 原始代码 | clean_code | 变化说明 |
|----------|-----------|----------|
| `utils/util.py` eval_metrix | `06_eval.py` eval_metrix | 完全一致（同样用 sklearn.metrics） |
| `results analysis/XJTU results.py` | `06_eval.py` eval_per_battery | 提取核心拆分逻辑，去掉了 Excel 导出 |
| 论文 Figure 4 | `plot_pred_vs_true` + `plot_degradation_curves` | 独立函数，不依赖 scienceplots |

---

> **模块 6 总结**：评估是检验整个 PINN 复现流程的最后一步。MAE/RMSE 给数字，散点图看整体，退化曲线看细节。如果 Figure 4a 的点在 y=x 附近、Figure 4b 的线能跟踪真实退化轨迹，复现流程就是正确的——不一定数值完全一样（超参、数据划分不同），但退化趋势和物理行为应该一致。